# 象棋 AI Phase 2: AlphaZero 自我对弈
**先跑完 Phase 1（棋谱预训练），再跑这个**

1. Runtime → T4 GPU
2. 挂载 Drive（checkpoint 存在 Drive 上，断了可以继续）
3. Run All

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR='/content/drive/MyDrive/xiangqi_ai'
CKPT_DIR=f'{SAVE_DIR}/checkpoints'
SP_DIR=f'{SAVE_DIR}/selfplay'
os.makedirs(CKPT_DIR,exist_ok=True)
os.makedirs(SP_DIR,exist_ok=True)
print('Drive mounted')

In [ ]:
%%writefile game.py
from __future__ import annotations
from dataclasses import dataclass
from enum import IntEnum
from typing import Optional
class PieceType(IntEnum):
    GENERAL=0;ADVISOR=1;ELEPHANT=2;HORSE=3;CHARIOT=4;CANNON=5;SOLDIER=6
class Color(IntEnum):
    RED=0;BLACK=1
    def opposite(self): return Color.BLACK if self==Color.RED else Color.RED
@dataclass
class Piece:
    type:PieceType;color:Color
@dataclass
class Move:
    from_pos:tuple;to_pos:tuple;captured_piece:Optional[Piece]=None
_O=((-1,0),(1,0),(0,-1),(0,1));_D=((-1,-1),(-1,1),(1,-1),(1,1))
_HM=((-2,-1,-1,0),(-2,1,-1,0),(-1,-2,0,-1),(-1,2,0,1),(1,-2,0,-1),(1,2,0,1),(2,-1,1,0),(2,1,1,0))
_EM=((-2,-2,-1,-1),(-2,2,-1,1),(2,-2,1,-1),(2,2,1,1))
_I=((0,0,4,1),(0,1,3,1),(0,2,2,1),(0,3,1,1),(0,4,0,1),(0,5,1,1),(0,6,2,1),(0,7,3,1),(0,8,4,1),(2,1,5,1),(2,7,5,1),(3,0,6,1),(3,2,6,1),(3,4,6,1),(3,6,6,1),(3,8,6,1),(6,0,6,0),(6,2,6,0),(6,4,6,0),(6,6,6,0),(6,8,6,0),(7,1,5,0),(7,7,5,0),(9,0,4,0),(9,1,3,0),(9,2,2,0),(9,3,1,0),(9,4,0,0),(9,5,1,0),(9,6,2,0),(9,7,3,0),(9,8,4,0))
class ChineseChessGame:
    __slots__=('board','current_player','move_history','_rp','_bp','_gp')
    def __init__(s):s.board={};s.current_player=Color.RED;s.move_history=[];s._rp=set();s._bp=set();s._gp={};s.reset()
    def reset(s):
        s.board.clear();s._rp.clear();s._bp.clear();s._gp.clear();s.current_player=Color.RED;s.move_history.clear()
        for r,c,t,co in _I:
            p=(r,c);s.board[p]=Piece(PieceType(t),Color(co));(s._rp if co==0 else s._bp).add(p)
            if t==0:s._gp[Color(co)]=p
    def _ps(s,c):return s._rp if c==Color.RED else s._bp
    def get_legal_moves(s,color=None):
        if color is None:color=s.current_player
        ps=[];b=s.board
        for pos in list(s._rp if color==Color.RED else s._bp):s._gpm(pos[0],pos[1],b[pos],ps)
        lg=[]
        for m in ps:
            s._do(m)
            if not s._chk(color) and not s._gf():lg.append(m)
            s._un(m)
        return lg
    def _gpm(s,r,c,p,ms):
        t=int(p.type);co=p.color;fp=(r,c);b=s.board
        if t==0:
            rm=7 if co==Color.RED else 0;rx=9 if co==Color.RED else 2
            for dr,dc in _O:
                nr,nc=r+dr,c+dc
                if rm<=nr<=rx and 3<=nc<=5:
                    tg=b.get((nr,nc))
                    if tg is None or tg.color!=co:ms.append(Move(fp,(nr,nc),tg))
        elif t==1:
            rm=7 if co==Color.RED else 0;rx=9 if co==Color.RED else 2
            for dr,dc in _D:
                nr,nc=r+dr,c+dc
                if rm<=nr<=rx and 3<=nc<=5:
                    tg=b.get((nr,nc))
                    if tg is None or tg.color!=co:ms.append(Move(fp,(nr,nc),tg))
        elif t==2:
            am=5 if co==Color.RED else 0;ax=9 if co==Color.RED else 4
            for dr,dc,br,bc in _EM:
                nr,nc=r+dr,c+dc
                if 0<=nr<=9 and 0<=nc<=8 and am<=nr<=ax and b.get((r+br,c+bc)) is None:
                    tg=b.get((nr,nc))
                    if tg is None or tg.color!=co:ms.append(Move(fp,(nr,nc),tg))
        elif t==3:
            for dr,dc,br,bc in _HM:
                nr,nc=r+dr,c+dc
                if 0<=nr<=9 and 0<=nc<=8 and b.get((r+br,c+bc)) is None:
                    tg=b.get((nr,nc))
                    if tg is None or tg.color!=co:ms.append(Move(fp,(nr,nc),tg))
        elif t==4:
            for dr,dc in _O:
                nr,nc=r+dr,c+dc
                while 0<=nr<=9 and 0<=nc<=8:
                    tg=b.get((nr,nc))
                    if tg is None:ms.append(Move(fp,(nr,nc),None))
                    else:
                        if tg.color!=co:ms.append(Move(fp,(nr,nc),tg))
                        break
                    nr+=dr;nc+=dc
        elif t==5:
            for dr,dc in _O:
                nr,nc=r+dr,c+dc;j=False
                while 0<=nr<=9 and 0<=nc<=8:
                    tg=b.get((nr,nc))
                    if not j:
                        if tg is None:ms.append(Move(fp,(nr,nc),None))
                        else:j=True
                    else:
                        if tg is not None:
                            if tg.color!=co:ms.append(Move(fp,(nr,nc),tg))
                            break
                    nr+=dr;nc+=dc
        elif t==6:
            fw=-1 if co==Color.RED else 1;cr=(r<=4) if co==Color.RED else(r>=5)
            nr=r+fw
            if 0<=nr<=9:
                tg=b.get((nr,c))
                if tg is None or tg.color!=co:ms.append(Move(fp,(nr,c),tg))
            if cr:
                for dc in(-1,1):
                    nc=c+dc
                    if 0<=nc<=8:
                        tg=b.get((r,nc))
                        if tg is None or tg.color!=co:ms.append(Move(fp,(r,nc),tg))
    def make_move(s,m):s._do(m);s.move_history.append(m);s.current_player=s.current_player.opposite()
    def _do(s,m):
        p=s.board[m.from_pos];del s.board[m.from_pos];ps=s._ps(p.color);ps.discard(m.from_pos)
        if m.captured_piece:s._ps(m.captured_piece.color).discard(m.to_pos)
        s.board[m.to_pos]=p;ps.add(m.to_pos)
        if p.type==PieceType.GENERAL:s._gp[p.color]=m.to_pos
    def _un(s,m):
        p=s.board[m.to_pos];ps=s._ps(p.color);del s.board[m.to_pos];ps.discard(m.to_pos)
        s.board[m.from_pos]=p;ps.add(m.from_pos)
        if m.captured_piece:s.board[m.to_pos]=m.captured_piece;s._ps(m.captured_piece.color).add(m.to_pos)
        if p.type==PieceType.GENERAL:s._gp[p.color]=m.from_pos
    def _chk(s,color):
        gp=s._gp.get(color)
        if gp is None:return True
        gr,gc=gp;opp=color.opposite();b=s.board
        for dr,dc in _O:
            nr,nc=gr+dr,gc+dc;j=False
            while 0<=nr<=9 and 0<=nc<=8:
                p=b.get((nr,nc))
                if p:
                    if not j:
                        if p.color==opp and p.type in(PieceType.CHARIOT,PieceType.GENERAL):return True
                        j=True
                    else:
                        if p.color==opp and p.type==PieceType.CANNON:return True
                        break
                nr+=dr;nc+=dc
        for dr,dc,br,bc in _HM:
            nr,nc=gr+dr,gc+dc
            if 0<=nr<=9 and 0<=nc<=8:
                p=b.get((nr,nc))
                if p and p.color==opp and p.type==PieceType.HORSE:
                    if abs(dr)==2:br2,bc2=nr+(-1 if dr>0 else 1),nc
                    else:br2,bc2=nr,nc+(-1 if dc>0 else 1)
                    if b.get((br2,bc2)) is None:return True
        if opp==Color.BLACK:
            p=b.get((gr-1,gc))
            if p and p.color==opp and p.type==PieceType.SOLDIER:return True
            for d in(-1,1):
                nc2=gc+d
                if 0<=nc2<=8:
                    p=b.get((gr,nc2))
                    if p and p.color==opp and p.type==PieceType.SOLDIER and gr>=5:return True
        else:
            p=b.get((gr+1,gc))
            if p and p.color==opp and p.type==PieceType.SOLDIER:return True
            for d in(-1,1):
                nc2=gc+d
                if 0<=nc2<=8:
                    p=b.get((gr,nc2))
                    if p and p.color==opp and p.type==PieceType.SOLDIER and gr<=4:return True
        return False
    def _gf(s):
        rp=s._gp.get(Color.RED);bp=s._gp.get(Color.BLACK)
        if not rp or not bp or rp[1]!=bp[1]:return False
        for r in range(min(rp[0],bp[0])+1,max(rp[0],bp[0])):
            if(r,rp[1])in s.board:return False
        return True
    def is_game_over(s):
        if not s.get_legal_moves(s.current_player):return True,s.current_player.opposite()
        if s._gf():return True,s.current_player
        return False,None
    def clone(s):
        g=ChineseChessGame.__new__(ChineseChessGame);g.board={p:Piece(v.type,v.color) for p,v in s.board.items()}
        g.current_player=s.current_player;g.move_history=[Move(m.from_pos,m.to_pos,Piece(m.captured_piece.type,m.captured_piece.color) if m.captured_piece else None) for m in s.move_history]
        g._rp=set(s._rp);g._bp=set(s._bp);g._gp=dict(s._gp);return g

In [ ]:
%%writefile encoding.py
import torch,numpy as np
from typing import Dict,List,Tuple
ROWS=10;COLS=9;NUM_SQUARES=90;NUM_PIECE_TYPES=7;NUM_INPUT_CHANNELS=15
def _rc(r,c):return r*COLS+c
def _pos(p):return p//COLS,p%COLS
def _ib(r,c):return 0<=r<ROWS and 0<=c<COLS
def _gen():
    s=set()
    for r in range(ROWS):
        for c in range(COLS):
            fp=_rc(r,c)
            for dr,dc in[(0,1),(0,-1),(1,0),(-1,0)]:
                for st in range(1,max(ROWS,COLS)):
                    nr,nc=r+dr*st,c+dc*st
                    if _ib(nr,nc):s.add((fp,_rc(nr,nc)))
            for dr,dc in[(-2,-1),(-2,1),(-1,-2),(-1,2),(1,-2),(1,2),(2,-1),(2,1)]:
                nr,nc=r+dr,c+dc
                if _ib(nr,nc):s.add((fp,_rc(nr,nc)))
    for r,c in[(0,3),(0,5),(1,4),(2,3),(2,5),(7,3),(7,5),(8,4),(9,3),(9,5)]:
        fp=_rc(r,c)
        for dr,dc in[(-1,-1),(-1,1),(1,-1),(1,1)]:
            nr,nc=r+dr,c+dc
            if _ib(nr,nc) and((0<=nr<=2 and 3<=nc<=5)or(7<=nr<=9 and 3<=nc<=5)):s.add((fp,_rc(nr,nc)))
    for r,c in[(0,2),(0,6),(2,0),(2,4),(2,8),(4,2),(4,6),(5,2),(5,6),(7,0),(7,4),(7,8),(9,2),(9,6)]:
        fp=_rc(r,c)
        for dr,dc in[(-2,-2),(-2,2),(2,-2),(2,2)]:
            nr,nc=r+dr,c+dc
            if _ib(nr,nc) and((r<=4 and nr<=4)or(r>=5 and nr>=5)):s.add((fp,_rc(nr,nc)))
    ml=sorted(s);return ml,{m:i for i,m in enumerate(ml)}
ALL_MOVES,MOVE_TO_INDEX=_gen()
NUM_ACTIONS=len(ALL_MOVES)
def move_to_index(f,t):return MOVE_TO_INDEX[(f,t)]
def index_to_move(i):return ALL_MOVES[i]
def board_to_tensor(bs,cp=0):
    t=torch.zeros(NUM_INPUT_CHANNELS,ROWS,COLS,dtype=torch.float32)
    for pos,(co,pt) in bs.items():r,c=_pos(pos);t[co*NUM_PIECE_TYPES+pt,r,c]=1.0
    if cp==0:t[14,:,:]=1.0
    return t

In [ ]:
%%writefile model.py
import torch,torch.nn as nn,torch.nn.functional as F
from encoding import NUM_INPUT_CHANNELS,NUM_ACTIONS,ROWS,COLS
class ResBlock(nn.Module):
    def __init__(s,ch):super().__init__();s.c1=nn.Conv2d(ch,ch,3,padding=1,bias=False);s.b1=nn.BatchNorm2d(ch);s.c2=nn.Conv2d(ch,ch,3,padding=1,bias=False);s.b2=nn.BatchNorm2d(ch)
    def forward(s,x):return F.relu(s.b2(s.c2(F.relu(s.b1(s.c1(x)))))+x)
class ChineseChessNet(nn.Module):
    def __init__(s,nf=128,nb=6):
        super().__init__();s.num_filters=nf;s.bs=ROWS*COLS
        s.ci=nn.Conv2d(NUM_INPUT_CHANNELS,nf,3,padding=1,bias=False);s.bi=nn.BatchNorm2d(nf)
        s.res=nn.Sequential(*[ResBlock(nf) for _ in range(nb)])
        s.pc=nn.Conv2d(nf,2,1,bias=False);s.pb=nn.BatchNorm2d(2);s.pf=nn.Linear(2*s.bs,NUM_ACTIONS)
        s.vc=nn.Conv2d(nf,1,1,bias=False);s.vb=nn.BatchNorm2d(1);s.vf1=nn.Linear(s.bs,128);s.vf2=nn.Linear(128,1)
    def forward(s,x):
        o=F.relu(s.bi(s.ci(x)));o=s.res(o)
        p=F.relu(s.pb(s.pc(o)));p=s.pf(p.view(p.size(0),-1))
        v=F.relu(s.vb(s.vc(o)));v=torch.tanh(s.vf2(F.relu(s.vf1(v.view(v.size(0),-1)))))
        return p,v
    def predict(s,bt):
        w=s.training;s.eval()
        if bt.dim()==3:bt=bt.unsqueeze(0)
        with torch.no_grad():p,v=s.forward(bt)
        if w:s.train()
        return p,v
def create_model(nf=128,nb=6):return ChineseChessNet(nf,nb)

In [ ]:
%%writefile selfplay.py
"""Self-play + training for AlphaZero Phase 2."""
import math,random,time,pickle,os,gc
import numpy as np,torch,torch.nn.functional as F,torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from game import ChineseChessGame,Color,Move
from encoding import board_to_tensor,move_to_index,NUM_ACTIONS,COLS,ROWS
from model import create_model

def _bs(g):
    return{r*COLS+c:(int(p.color),int(p.type))for(r,c),p in g.board.items()}

def _mai(m):
    return move_to_index(m.from_pos[0]*COLS+m.from_pos[1],m.to_pos[0]*COLS+m.to_pos[1])

# === MCTS ===
class Node:
    __slots__=('parent','children','move','n','w','p')
    def __init__(s,parent=None,move=None,prior=0.0):
        s.parent=parent;s.children={};s.move=move;s.n=0;s.w=0.0;s.p=prior
    def ucb(s,cpuct):
        pn=s.parent.n if s.parent else 1
        return(s.w/max(1,s.n))+cpuct*s.p*math.sqrt(pn)/(1+s.n)

def mcts_search(model,game,device,n_sim=200,cpuct=1.5,temp=1.0):
    root=Node()
    legal=game.get_legal_moves()
    if not legal:return[],np.array([])
    # Expand root
    pol,val=_evaluate(model,game,device)
    _expand(root,legal,pol)
    # Dirichlet noise
    if root.children:
        noise=np.random.dirichlet([0.3]*len(root.children))
        for i,c in enumerate(root.children.values()):c.p=0.75*c.p+0.25*noise[i]
    root.n=1;root.w=val
    # Simulate
    for _ in range(n_sim):
        node=root;g2=game.clone();path=[node]
        while node.children:
            best=max(node.children.values(),key=lambda c:c.ucb(cpuct))
            node=best;path.append(node);g2.make_move(node.move)
        over,winner=g2.is_game_over()
        if over:
            if winner is None:v=0.0
            else:
                jm=g2.current_player.opposite()
                v=1.0 if winner==jm else -1.0
        else:
            lm=g2.get_legal_moves()
            if not lm:v=0.0
            else:
                p2,v2=_evaluate(model,g2,device)
                _expand(node,lm,p2);v=-v2
        for nd in reversed(path):nd.n+=1;nd.w+=v;v=-v
    # Collect
    moves=[];counts=[]
    for ai,ch in root.children.items():moves.append(ch.move);counts.append(ch.n)
    counts=np.array(counts,dtype=np.float64)
    if temp<1e-4:
        probs=np.zeros_like(counts);probs[np.argmax(counts)]=1.0
    else:
        ct=counts**(1.0/temp);probs=ct/ct.sum()
    return moves,probs

def _evaluate(model,game,device):
    bs=_bs(game);cp=0 if game.current_player==Color.RED else 1
    t=board_to_tensor(bs,cp).to(device)
    pl,vt=model.predict(t);pl=pl.squeeze(0)
    legal=game.get_legal_moves();li=set()
    for m in legal:
        try:li.add(_mai(m))
        except:pass
    mask=torch.full((NUM_ACTIONS,),float('-inf'),device=device)
    for i in li:mask[i]=0.0
    pol=F.softmax(pl+mask,dim=0).cpu().numpy()
    return pol,vt.item()

def _expand(node,legal,policy):
    ais=[];prs=[]
    for m in legal:
        try:ai=_mai(m);ais.append(ai);prs.append(policy[ai])
        except:continue
    s=sum(prs)
    if s>1e-8:prs=[p/s for p in prs]
    else:prs=[1.0/len(prs)]*len(prs)
    for i,(ai,m) in enumerate(zip(ais,legal)):
        if ai in node.children:continue
        node.children[ai]=Node(parent=node,move=m,prior=prs[i])

# === Self-play game ===
def play_one_game(model,device,n_sim=200,temp_thresh=30,max_moves=200):
    game=ChineseChessGame();positions=[];mc=0
    while mc<max_moves:
        over,winner=game.is_game_over()
        if over:break
        temp=1.0 if mc<temp_thresh else 0.1
        moves,probs=mcts_search(model,game,device,n_sim=n_sim,temp=temp)
        if not moves:break
        # Record
        bs=_bs(game);cp=0 if game.current_player==Color.RED else 1
        bt=board_to_tensor(bs,cp).numpy()
        pt=np.zeros(NUM_ACTIONS,dtype=np.float32)
        for m,p in zip(moves,probs):
            try:pt[_mai(m)]=p
            except:pass
        positions.append((bt,pt,cp))
        # Play
        mi=np.random.choice(len(moves),p=probs)
        game.make_move(moves[mi]);mc+=1
    if not game.is_game_over()[0]:winner=None
    else:_,winner=game.is_game_over()
    wi=int(winner) if winner is not None else None
    examples=[]
    for bt,pt,cp in positions:
        if wi is None:v=0.0
        elif cp==wi:v=1.0
        else:v=-1.0
        examples.append((bt,pt,v))
    return examples,wi,mc

# === Training ===
class SPDS(Dataset):
    def __init__(s,data):s.data=data
    def __len__(s):return len(s.data)
    def __getitem__(s,i):
        bt,pt,v=s.data[i]
        return torch.from_numpy(bt),torch.from_numpy(pt),torch.tensor(v,dtype=torch.float32)

def train_on_examples(model,examples,device,epochs=5,bs=256,lr=0.001):
    model.to(device);model.train()
    opt=optim.Adam(model.parameters(),lr=lr,weight_decay=1e-4)
    ds=SPDS(examples);dl=DataLoader(ds,batch_size=bs,shuffle=True,num_workers=2,pin_memory=True)
    for ep in range(epochs):
        tl=tp=tv=nb=0
        for b,p,v in dl:
            b,p,v=b.to(device),p.to(device),v.to(device)
            pl,pv=model(b);pv=pv.squeeze(-1)
            ploss=-torch.sum(p*F.log_softmax(pl,1),1).mean();vloss=F.mse_loss(pv,v)
            loss=ploss+vloss;opt.zero_grad();loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step()
            tp+=ploss.item();tv+=vloss.item();nb+=1
        print(f'  Train ep{ep+1}: loss={tp/nb+tv/nb:.4f}',flush=True)
    return model

## 加载 Phase 1 模型 + 开始自我对弈

In [ ]:
from model import create_model
from selfplay import play_one_game, train_on_examples
import torch, os, pickle, time, random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = create_model(128, 6)

# Load Phase 1 or latest Phase 2 checkpoint
SP_BEST = f'{CKPT_DIR}/selfplay_best.pt'
P1_BEST = f'{CKPT_DIR}/best.pt'
start_iter = 1

if os.path.exists(SP_BEST):
    ckpt = torch.load(SP_BEST, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    start_iter = ckpt.get('iteration', 0) + 1
    print(f'Resumed Phase 2 from iteration {start_iter-1}')
elif os.path.exists(P1_BEST):
    ckpt = torch.load(P1_BEST, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded Phase 1 model (epoch {ckpt.get("epoch","?")})')
else:
    print('No checkpoint found, training from scratch')

model.to(device).eval()
print(f'Device: {device}, starting from iteration {start_iter}')

In [ ]:
# === AlphaZero Self-Play Loop ===
# Settings
NUM_ITERATIONS = 50      # total iterations (can resume)
GAMES_PER_ITER = 10      # self-play games per iteration
MCTS_SIMS = 200          # MCTS simulations per move
TRAIN_EPOCHS = 5         # training epochs per iteration
BUFFER_SIZE = 100000     # replay buffer capacity

# Load or create replay buffer
BUF_PATH = f'{SP_DIR}/replay_buffer.pkl'
if os.path.exists(BUF_PATH):
    with open(BUF_PATH, 'rb') as f:
        replay_buffer = pickle.load(f)
    print(f'Loaded replay buffer: {len(replay_buffer)} examples')
else:
    replay_buffer = []
    print('New replay buffer')

for iteration in range(start_iter, NUM_ITERATIONS + 1):
    t0 = time.time()
    print(f'\n=== Iteration {iteration}/{NUM_ITERATIONS} ===', flush=True)

    # 1. Self-play
    model.eval()
    iter_examples = []
    rw = bw = dr = 0
    for gi in range(GAMES_PER_ITER):
        gt0 = time.time()
        exs, winner, nmoves = play_one_game(model, device, n_sim=MCTS_SIMS)
        iter_examples.extend(exs)
        w_str = 'Red' if winner == 0 else ('Black' if winner == 1 else 'Draw')
        if winner == 0: rw += 1
        elif winner == 1: bw += 1
        else: dr += 1
        print(f'  Game {gi+1}/{GAMES_PER_ITER}: {nmoves} moves, {w_str} [{time.time()-gt0:.0f}s]', flush=True)

    print(f'Self-play: {len(iter_examples)} positions, R={rw} B={bw} D={dr}', flush=True)

    # 2. Add to replay buffer (keep latest BUFFER_SIZE)
    replay_buffer.extend(iter_examples)
    if len(replay_buffer) > BUFFER_SIZE:
        replay_buffer = replay_buffer[-BUFFER_SIZE:]
    print(f'Replay buffer: {len(replay_buffer)}', flush=True)

    # 3. Train
    random.shuffle(replay_buffer)
    train_data = replay_buffer[-min(50000, len(replay_buffer)):]
    model = train_on_examples(model, train_data, device, epochs=TRAIN_EPOCHS)

    # 4. Save checkpoint
    torch.save({
        'iteration': iteration,
        'model_state_dict': model.state_dict(),
    }, SP_BEST)
    # Save buffer every 5 iterations
    if iteration % 5 == 0:
        with open(BUF_PATH, 'wb') as f:
            pickle.dump(replay_buffer, f, protocol=4)
        print(f'  Buffer saved to Drive', flush=True)

    elapsed = time.time() - t0
    print(f'Iteration {iteration} done in {elapsed:.0f}s', flush=True)

# Final save
with open(BUF_PATH, 'wb') as f:
    pickle.dump(replay_buffer, f, protocol=4)
print('\nPhase 2 training complete!')

## 下载模型

In [ ]:
from google.colab import files
if os.path.exists(SP_BEST):
    files.download(SP_BEST)
    print('Downloaded selfplay_best.pt')
print('Model also saved to Google Drive/xiangqi_ai/checkpoints/')